# FLUKE Named Entity Recognition with DeepSeek R1

This notebook evaluates NER robustness using DeepSeek R1 open-source reasoning model via OpenRouter API with FLUKE linguistic modifications.

In [2]:
# Standard imports
from datasets import load_dataset
import dspy
import os
import pandas as pd
import json
import glob
import time
import ast
from dotenv import load_dotenv
from dspy.evaluate import Evaluate

# Import unified FLUKE utilities
from fluke_reasoning_utils import (
    REASONING_MODELS, REASONING_CONFIGS,
    remove_space, extract_ner_prediction,
    aggregate_results, highlight_drops_and_significance,
    compare_models, calculate_f1_ent, convert_string_to_entities
)

/Users/hungthinh/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load environment variables
load_dotenv()

# For OpenRouter, we need the OpenRouter API key
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
if not openrouter_api_key:
    print("Warning: OPENROUTER_API_KEY not found in environment variables")
    print("Please set your OpenRouter API key in the .env file")

## DeepSeek R1 Configuration

In [4]:
# Select DeepSeek configuration
CONFIG_NAME = 'deepseek'  # Options: 'deepseek', 'deepseek-lite'
config = REASONING_CONFIGS[CONFIG_NAME]

MODEL_NAME = config['model']
MODEL_ID = REASONING_MODELS[MODEL_NAME]

print(f"Configuration: {CONFIG_NAME}")
print(f"Model: {MODEL_NAME} ({MODEL_ID})")
print(f"Description: {config['description']}")

# Configure DSPy with DeepSeek R1 via OpenRouter
lm = dspy.LM(
    model=MODEL_ID,
    api_key=openrouter_api_key,
    api_base="https://openrouter.ai/api/v1",
    max_tokens=20_000,
    temperature=1  # DeepSeek R1 supports temperature control
)
dspy.configure(lm=lm)

Configuration: deepseek
Model: deepseek-r1 (openrouter/deepseek/deepseek-r1)
Description: Open-source reasoning with DeepSeek R1


## Load NER Data

In [5]:
# Load NER dataset
ds = pd.read_json('../../../data/train_dev_test_data/ner/fewnerd_sample_test.json', encoding_errors='replace')
ds = ds.to_dict('records')

print(f"Loaded {len(ds)} NER samples")
print(f"Sample structure: {list(ds[0].keys())}")

# Create examples
examples = [
    dspy.Example({
        "text": r["text"],
        "label": str(r['label'])
    }).with_inputs("text")
    for r in ds
]

# Test example
example = examples[0]
print(f"\nExample text: {example.text}")
print(f"Label: {example.label}")

Loaded 249 NER samples
Sample structure: ['id', 'text', 'label', 'dataset', 'entity']

Example text: In the early 1930s the band moved to the Grill Room of the Taft Hotel in New York ; the band was renamed ``George Hall and His Hotel Taft Orchestra``.
Label: [{'Grill Room': 'BUILDING'}, {'Taft Hotel': 'BUILDING'}, {'New York': 'LOCATION'}, {'George Hall and His Hotel Taft Orchestra': 'ORGANIZATION'}]


## Define Task with DeepSeek R1

In [6]:
class DeepSeekEnt(dspy.Signature):
    """Extract named entities from the text. Think step by step about entity boundaries, types, and context. Possible entity types: ART, BUILDING, EVENT, LOCATION, ORGANIZATION, OTHER, PERSON, PRODUCT."""
    text = dspy.InputField()
    label = dspy.OutputField(desc="The list of named entities in the text: [{'text': the text span, 'value': the entity label},].", prefix='Entities:')

class DeepSeekEntModule(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.Predict(DeepSeekEnt)

    def forward(self, text):
        return self.prog(text=text)

# Initialize module
deepseek_ent = DeepSeekEntModule()

# Evaluation metric
def eval_metric(true, prediction, trace=None):
    pred = prediction.label
    parsed_answer = extract_ner_prediction(pred)
    if not parsed_answer:
        return 0.0
    gold_entities = ast.literal_eval(true.label)
    precision, recall, f1_score = calculate_f1_ent(gold_entities, parsed_answer)
    return f1_score

In [7]:
# Test single example
pred = deepseek_ent(text=example.text)
print(f"Text: {example.text}")
print(f"True Entities: {example.label}")
print(f"Prediction: {pred.label}")
print(f"F1 Score: {eval_metric(example, pred):.3f}")

Text: In the early 1930s the band moved to the Grill Room of the Taft Hotel in New York ; the band was renamed ``George Hall and His Hotel Taft Orchestra``.
True Entities: [{'Grill Room': 'BUILDING'}, {'Taft Hotel': 'BUILDING'}, {'New York': 'LOCATION'}, {'George Hall and His Hotel Taft Orchestra': 'ORGANIZATION'}]
Prediction: [{'text': 'Grill Room', 'value': 'BUILDING'}, {'text': 'Taft Hotel', 'value': 'BUILDING'}, {'text': 'New York', 'value': 'LOCATION'}, {'text': 'George Hall', 'value': 'PERSON'}, {'text': 'Hotel Taft', 'value': 'BUILDING'}, {'text': 'George Hall and His Hotel Taft Orchestra', 'value': 'ORGANIZATION'}]
F1 Score: 0.800


## Evaluate Original Dataset

In [8]:
# Test size for DeepSeek R1
TEST_SIZE = 200  # Adjust based on API limits and budget
test_examples = examples

print(f"Evaluating {len(test_examples)} examples with DeepSeek R1...")

evaluate = Evaluate(
    devset=test_examples,
    metric=eval_metric,
    num_threads=2,  # Moderate threading for OpenRouter
    display_progress=True,
    display_table=10,
    return_all_scores=True
)

results = evaluate(deepseek_ent)

# Save results
items = []
for sample in results['results']:
    pred = sample[1].get('label', '[]') if sample[1] != {} else '[]'
    items.append({
        'text': sample[0]['text'],
        'label': sample[0]['label'],
        'pred': pred,
        'raw_output': pred
    })

df_result = pd.DataFrame(items)
output_file = f'../results/ner/{MODEL_NAME}-{CONFIG_NAME}-0shot-ner.csv'
df_result.to_csv(output_file, index=False)

print(f"\nDeepSeek R1 Average F1 Score: {results['score']:.3f}")
print(f"Results saved to: {output_file}")

Evaluating 249 examples with DeepSeek R1...
Average Metric: 134.44 / 249 (54.0%): 100%|██████████| 249/249 [31:43<00:00,  7.64s/it] 

2025/08/18 17:06:42 INFO dspy.evaluate.evaluate: Average Metric: 134.44064660829366 / 249 (54.0%)


,text,example_label,pred_label,eval_metric
0,In the early 1930s the band moved to the Grill Room of the Taft Ho...,"[{'Grill Room': 'BUILDING'}, {'Taft Hotel': 'BUILDING'}, {'New Yor...","[{'text': 'Grill Room', 'value': 'BUILDING'}, {'text': 'Taft Hotel...",✔️ [0.800]
1,The final season of minor league play Elkin Memorial Park saw seas...,[{'Elkin Memorial Park': 'LOCATION'}],"[{'text': 'Elkin Memorial Park', 'value': 'LOCATION'}]",✔️ [1.000]
2,"They finished the season 14\u201319, 9\u20139 in C-USA play to fin...",[{'C-USA play': 'EVENT'}],"[{'text': 'C-USA', 'value': 'ORGANIZATION'}]",
3,"The B-52 pilot, Major Larry G.Messinger, later recalled,","[{'B-52': 'PRODUCT'}, {'Larry G.Messinger': 'PERSON'}]","[{'text': 'B-52', 'value': 'PRODUCT'}, {'text': 'Larry G. Messinge...",✔️ [0.500]
4,The Austro-Hungarian Navy built and operated two classes of protec...,[{'Austro-Hungarian Navy': 'ORGANIZATION'}],"[{'text': 'Austro-Hungarian Navy', 'value': 'ORGANIZATION'}, {'tex...",✔️ [0.667]
5,Elin Hilderbrand is an American writer mostly of romance novels.,"[{'Elin Hilderbrand': 'PERSON'}, {'American': 'LOCATION'}]","[\n{'text': 'Elin Hilderbrand', 'value': 'PERSON'}\n]",
6,A prototype was fitted in the mid-'60s in a one-off DB5 extended 4...,"[{""DB5 extended 4''"": 'PRODUCT'}, {'Marek': 'PERSON'}, {'Aston Mar...","[{'text': 'DB5', 'value': 'PRODUCT'}, {'text': 'Marek', 'value': '...",✔️ [0.250]
7,He has caught the attention of major publications and media outlet...,"[{'CNN': 'ORGANIZATION'}, {'The Huffington Post': 'ORGANIZATION'},...","[{'text': 'CNN', 'value': 'ORGANIZATION'}, {'text': 'The Huffingto...",✔️ [0.824]
8,The Cnidaria are a group of animals found exclusively in aquatic a...,[{'Cnidaria': 'OTHER'}],"[{'text': 'Cnidaria', 'value': 'OTHER'}]",✔️ [1.000]
9,The Ninth suffered a serious defeat at the Battle of Camulodunum u...,"[{'Camulodunum': 'EVENT'}, {'Quintus Petillius Cerialis': 'PERSON'...","[ {'text': 'The Ninth', 'value': 'ORGANIZATION'}, {'text': 'Battle...",



DeepSeek R1 Average F1 Score: 53.990
Results saved to: ../results/ner/deepseek-r1-deepseek-0shot-ner.csv


## Evaluate Modifications

In [14]:
def evaluate_modified_set(data, program, max_samples=50):
    """Evaluate on modified dataset with DeepSeek R1."""
    limited_data = data[:max_samples] if len(data) > max_samples else data
    
    mod_examples = [
        dspy.Example({
            "text": remove_space(r["modified_text"]),
            "label": str(r['modified_label'] if 'modified_label' in r else r['label']),
            "original_text": remove_space(r['original_text']),
            "original_label": str(r['original_label'] if 'original_label' in r else r['label']),
            "index": r.get('index', 0),
            "type": r.get('subtype', None)
        }).with_inputs("text")
        for r in limited_data
    ]
    
    evaluate = Evaluate(
        devset=mod_examples,
        metric=eval_metric,
        num_threads=2,  # Moderate threading for OpenRouter
        display_progress=True,
        display_table=1,
        return_all_scores=True,
        provide_traceback=True
    )
    
    return evaluate(program)

In [15]:
# Load original predictions
original_pred_file = f'../results/ner/{MODEL_NAME}-{CONFIG_NAME}-0shot-ner.csv'
if os.path.exists(original_pred_file):
    original_pred_ds = pd.read_csv(original_pred_file)
    original_pred_ds['text'] = original_pred_ds['text'].apply(lambda x: remove_space(x.encode('utf-8').decode('unicode-escape')))
    print(f"Loaded original DeepSeek R1 predictions from {original_pred_file}")
else:
    print("Please run original evaluation first")
    original_pred_ds = None

# Test modifications with DeepSeek R1
json_files = glob.glob('../../../data/modified_data/ner/*_100.json')

print(f"\nTesting {len(json_files)} modifications with DeepSeek R1...")

for json_file in json_files:
    print(f"\nProcessing: {json_file.split('/')[-1]}")
    
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    # Evaluate with sample limit
    results_mod = evaluate_modified_set(data, deepseek_ent, max_samples=150)
    
    # Process results
    items = []
    for sample in results_mod['results']:
        pred = sample[1].get('label', '[]') if sample[1] != {} else '[]'
        pred_extracted = extract_ner_prediction(pred)
        
        item = {
            'text': sample[0]['text'],
            'original_text': sample[0]['original_text'].encode('utf-8').decode('unicode-escape'),
            'modified_label': sample[0]['label'],
            'original_label': sample[0]['original_label'],
            'modified_pred': [{entity['text']: entity['value']} for entity in pred_extracted] if isinstance(pred_extracted, list) else [],
            'index': sample[0]['index'],
            'type': sample[0]['type'],
            'raw_output': pred
        }
        
        # Find original prediction
        if original_pred_ds is not None and item['index'] < len(original_pred_ds):
            item['original_pred'] = original_pred_ds['pred'].iloc[item['index']]
        else:
            item['original_pred'] = '[]'
        
        # Handle NaN in original_label
        if pd.isna(item['original_label']):
            item['original_label'] = item['modified_label']
        
        items.append(item)
    
    df_mod = pd.DataFrame(items)
    mod_name = json_file.split('/')[-1].replace('.json', '')
    output_file = f'../results/ner/{MODEL_NAME}-{CONFIG_NAME}-0shot-{mod_name}.csv'
    df_mod.to_csv(output_file, index=False)
    
    print(f"Average F1: {results_mod['score']:.3f}")
    print(f"Saved to: {output_file}")
    
    time.sleep(3)  # Rate limiting for OpenRouter

Loaded original DeepSeek R1 predictions from ../results/ner/deepseek-r1-deepseek-0shot-ner.csv

Testing 17 modifications with DeepSeek R1...

Processing: casual_100.json
Average Metric: 56.33 / 98 (57.5%): 100%|██████████| 98/98 [00:00<00:00, 282.27it/s]

2025/08/18 20:42:29 INFO dspy.evaluate.evaluate: Average Metric: 56.32572721396251 / 98 (57.5%)


,text,example_label,original_text,original_label,index,type,pred_label,eval_metric
0,'The Rake's Progress' is a '45 British comedy-drama flick.,"[{'text': ""The Rake's Progress"", 'value': 'ART'}, {'text': 'Britis...",The Rake's Progress is a 1945 British comedy-drama film.,"[{""The Rake's Progress"": 'ART'}, {'British': 'NATIONALITY'}]",0,None,"[{'text': 'The Rake\'s Progress', 'value': 'PRODUCT'}, {'text': 'B...",


Average F1: 57.480
Saved to: ../results/ner/deepseek-r1-deepseek-0shot-casual_100.csv

Processing: discourse_100.json
Average Metric: 37.99 / 72 (52.8%): 100%|██████████| 72/72 [00:00<00:00, 432.00it/s]

2025/08/18 20:42:32 INFO dspy.evaluate.evaluate: Average Metric: 37.99498834498834 / 72 (52.8%)


...

,text,example_label,original_text,original_label,index,type,pred_label,eval_metric
0,"Genre classics are focused on by Back to Basics, with older movies...","[{'text': 'Back to Basics', 'value': 'ORGANIZATION'}]","Back to Basics focuses on genre classics, showing older movies and...",[{'Back to Basics': 'ORGANIZATION'}],56,None,"[{'text': 'Back to Basics', 'value': 'ORGANIZATION'}]",✔️ [1.000]


Average F1: 57.830
Saved to: ../results/ner/deepseek-r1-deepseek-0shot-active_to_passive_100.csv

Processing: negation_100.json
Average Metric: 53.16 / 110 (48.3%): 100%|██████████| 110/110 [16:02<00:00,  8.75s/it]

2025/08/18 22:59:21 INFO dspy.evaluate.evaluate: Average Metric: 53.16191194426489 / 110 (48.3%)


,text,example_label,original_text,original_label,index,type,pred_label,eval_metric
0,"It is developed by no company, neither Golaem nor any other (creat...","[{'text': 'Golaem', 'value': 'ORGANIZATION'}, {'text': 'Rennes', '...","It is developed by Golaem, a France -based software company (creat...","[{'Golaem': 'ORGANIZATION'}, {'France': 'LOCATION'}, {'Rennes': 'L...",28,absolute,"[{'text': 'Golaem', 'value': 'ORGANIZATION'}, {'text': 'Rennes', '...",✔️ [0.800]


Average F1: 48.330
Saved to: ../results/ner/deepseek-r1-deepseek-0shot-negation_100.csv


## Chain-of-Thought with DeepSeek R1

In [ ]:
class CoTDeepSeekEnt(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.ChainOfThought(DeepSeekEnt)

    def forward(self, text):
        return self.prog(text=text)

# Test CoT
cot_deepseek_ent = CoTDeepSeekEnt()
pred_cot = cot_deepseek_ent(text=example.text)
print("Chain-of-Thought with DeepSeek R1:")
print(f"Text: {example.text}")
print(f"\nReasoning: {pred_cot.reasoning if hasattr(pred_cot, 'reasoning') else 'N/A'}")
print(f"\nPrediction: {pred_cot.label}")

## Aggregate Results

In [ ]:
# Aggregate all modification results
result_files = glob.glob(f'results/ner/{MODEL_NAME}-{CONFIG_NAME}-0shot-*_100.csv')

if result_files:
    results_df = aggregate_results(
        result_files,
        task_name='named_entity_recognition',
        model_name=f'{MODEL_NAME}-{CONFIG_NAME}'
    )
    
    if not results_df.empty:
        # Display summary
        print(f"\n{MODEL_NAME}-{CONFIG_NAME} Results Summary:")
        columns_to_show = ['modification', 'original_res', 'modified_res', 'difference', 'samples']
        if 'original_precision' in results_df.columns:
            columns_to_show.extend(['original_precision', 'original_recall', 'modified_precision', 'modified_recall'])
        print(results_df[columns_to_show])
        
        # Save aggregated results
        output_file = f'results/ner/{MODEL_NAME}-{CONFIG_NAME}-DP.csv'
        results_df.to_csv(output_file, index=False)
        print(f"\nAggregated results saved to: {output_file}")
        
        # Display styled results
        styled_df = results_df.round(3).style.apply(highlight_drops_and_significance, axis=1)
        display(styled_df)
else:
    print("No result files found to aggregate")

## Model Comparison

In [ ]:
# Compare DeepSeek R1 with other models
comparison_files = {
    'DeepSeek-R1': f'results/ner/{MODEL_NAME}-{CONFIG_NAME}-0shot-ner.csv',
    'GPT-5': 'results/ner/gpt-5-standard-0shot-ner.csv',
    'GPT-4o': 'results/ner/gpt4o-0shot-ner.csv',
    'Claude-3.5': 'results/ner/claude-0shot-ner.csv',
    'o3-2025-04-16': 'results/ner/o3-2025-04-16-standard-0shot-ner.csv',
    'Llama-405B': 'results/ner/llama-0shot-ner.csv'
}

comparison_df = compare_models(comparison_files, task_name='named_entity_recognition')

if not comparison_df.empty:
    print("\nModel Comparison (including DeepSeek R1):")
    print(comparison_df)
    
    # Calculate DeepSeek R1 performance relative to others
    if 'DeepSeek-R1' in comparison_df['Model'].values:
        deepseek_f1 = comparison_df[comparison_df['Model'] == 'DeepSeek-R1']['F1 Score'].values[0]
        
        # Compare with closed-source models
        closed_models = ['GPT-5', 'GPT-4o', 'Claude-3.5', 'o3-2025-04-16']
        closed_f1s = comparison_df[comparison_df['Model'].isin(closed_models)]['F1 Score'].values
        
        if len(closed_f1s) > 0:
            avg_closed = closed_f1s.mean()
            gap = deepseek_f1 - avg_closed
            print(f"\nDeepSeek R1 F1 Score: {deepseek_f1:.3f}")
            print(f"Average of closed-source models: {avg_closed:.3f}")
            print(f"Performance gap: {gap:+.3f} ({gap*100:+.1f}%)")
            print(f"\nNote: DeepSeek R1 is an open-source model competing with proprietary systems")
    
    # Highlight best performer
    def highlight_max(s):
        is_max = s == s.max()
        return ['background-color: green; color: white' if v else '' for v in is_max]
    
    styled_comparison = comparison_df.style.apply(highlight_max, subset=['F1 Score'])
    display(styled_comparison)
else:
    print("No comparison data available")

## DeepSeek R1 Performance Analysis

In [ ]:
print(f"\n{'='*60}")
print(f"FLUKE NER with DeepSeek R1 Complete!")
print(f"{'='*60}")

if 'results' in locals():
    print(f"\nBase F1 score: {results[0]:.3f}")

if 'results_df' in locals() and not results_df.empty:
    avg_row = results_df[results_df['modification'] == 'average'].iloc[0]
    print(f"Average robustness drop: {avg_row['difference']:.3f}")
    print(f"Modifications tested: {len(results_df) - 1}")
    
    if 'original_precision' in avg_row:
        print(f"\nDetailed metrics:")
        print(f"  Original Precision: {avg_row['original_precision']:.3f}")
        print(f"  Original Recall: {avg_row['original_recall']:.3f}")
        print(f"  Modified Precision: {avg_row['modified_precision']:.3f}")
        print(f"  Modified Recall: {avg_row['modified_recall']:.3f}")

print(f"\nDeepSeek R1 Configuration: {config['description']}")
print(f"\nKey advantages of DeepSeek R1:")
print("• Open-source model with competitive NER performance")
print("• Strong entity boundary detection")
print("• Good understanding of entity types")
print("• Cost-effective via OpenRouter API")
print("• Supports temperature control")

print(f"\nFiles saved in: results/ner/")